# Survival Analysis

This notebook contains the cluster summarization workflow.
Use it to build patient-level features for downstream survival analysis.


In [ ]:
from pathlib import Path
import importlib.util

module_path = Path('survival analysis.py').resolve()
if not module_path.exists():
    raise FileNotFoundError(f'Cannot find script: {module_path}')

spec = importlib.util.spec_from_file_location('survival_analysis_impl', module_path)
sa = importlib.util.module_from_spec(spec)
spec.loader.exec_module(sa)

# Expose frequently used functions
summarize_clusters_from_root = sa.summarize_clusters_from_root
merge_cluster_with_clinical = sa.merge_cluster_with_clinical
run_km_analyses = sa.run_km_analyses
run_cox_models_by_treatment = sa.run_cox_models_by_treatment
run_cox_diagnostics = sa.run_cox_diagnostics
run_univariate_cox_for_lasso_selected = sa.run_univariate_cox_for_lasso_selected
run_univariate_analysis_and_forest_plots = sa.run_univariate_analysis_and_forest_plots

In [ ]:
# Step 1: build cluster summary from discovered S2Omics outputs

demo_root = Path('../demo/global')
cluster_summary_path = demo_root / 'cluster_summary.csv'

df = summarize_clusters_from_root(
    root_dir=demo_root,
    output_path=cluster_summary_path,
    patch_size=16,
    pixel_size=0.5,
)
df.head()

In [ ]:
# Step 2: merge cluster summary with clinical table

clinical_path = demo_root / 'survival.txt'
merged_output_path = demo_root / 'merged_clinical_cluster_summary.csv'

merged_df = merge_cluster_with_clinical(
    clinical_path=clinical_path,
    cluster_summary_path=cluster_summary_path,
    output_path=merged_output_path,
)

preview_columns = [
    'patient_id',
    'cluster_patient_id',
    'clinical_patient_id',
    'OS_days',
    'OS_event',
    'PFS_days',
    'PFS_event',
    'Stage',
    'Age_dicho',
    'Sex',
    'smoking_status',
]
existing_columns = [col for col in preview_columns if col in merged_df.columns]
merged_df[existing_columns].head()

## Kaplan-Meier Curves by Broad_treatment and Legacy OS Subgroups

This section keeps the Broad_treatment Kaplan-Meier analysis for both endpoints and also preserves the older OS subgroup curves.

Broad_treatment endpoint plots:

- Progression-Free Survival (PFS)
  - Time: `PFS_days`
  - Event: `PFS_censoring` (`1 = progression event`, `0 = censored`)
- Overall Survival (OS)
  - Time: `OS_days`
  - Event: OS censoring column (`1 = death event`, `0 = censored`)

Legacy OS plots kept for comparison:

- Overall survival by Histology
- Overall survival by Stage
- Overall survival by Smoking

For each plot, a global log-rank test is reported when subgrouping is possible.


In [ ]:
# Step 3: Kaplan-Meier analyses

km_results, legacy_os_stats = run_km_analyses(merged_df, min_group_n=10)
km_results, legacy_os_stats

## Clinical Group Comparisons

This unified section compares outcomes and cluster features across clinical variables:

- Histology (`OS_days` box plots)
- Tissue origin
- Sample location within the Distant metastasis subgroup
- Stage
- Smoking

To protect sensitive data, these analyses report group-level summaries and test statistics only.


In [ ]:
# Step 5: clinical group comparison plots and statistics

def run_clinical_group_comparisons(merged_df):
    cluster_feature_cols = sa.get_cluster_feature_columns(merged_df)

    tissue_origin_col = sa._find_column(
        merged_df,
        ['Tissue_origin', 'tissue_origin', 'Tissue Origin', 'tissue origin'],
    )
    tissue_origin_stats = sa.compare_cluster_features_by_group(
        df=merged_df,
        group_col=tissue_origin_col,
        title='Tissue Origin',
        cluster_cols=cluster_feature_cols,
        min_group_n=5,
        top_n_features=6,
    )

    sample_location_col = sa._find_column(
        merged_df,
        ['Sample_location', 'sample_location', 'Sample Location', 'sample location'],
    )
    if tissue_origin_col is not None and sample_location_col is not None:
        distant_mask = merged_df[tissue_origin_col].astype(str).str.lower().str.contains('distant metastasis', na=False)
        distant_df = merged_df.loc[distant_mask].copy()
        distant_location_stats = sa.compare_cluster_features_by_group(
            df=distant_df,
            group_col=sample_location_col,
            title='Distant Metastasis Subgroup by Sample Location',
            cluster_cols=cluster_feature_cols,
            min_group_n=3,
            top_n_features=6,
        )
    else:
        print('[Distant Metastasis Subgroup by Sample Location] Skipped: Tissue_origin and/or Sample_location column not found.')
        distant_location_stats = None

    stage_col = sa._find_column(merged_df, ['Stage', 'stage', 'Clinical_stage', 'clinical_stage'])
    stage_stats = sa.compare_cluster_features_by_group(
        df=merged_df,
        group_col=stage_col,
        title='Stage',
        cluster_cols=cluster_feature_cols,
        min_group_n=5,
        top_n_features=6,
    )

    smoking_col = sa._find_column(
        merged_df,
        ['Smoking', 'smoking', 'smoking_status', 'Smoking_status', 'Smoking Status'],
    )
    smoking_stats = sa.compare_cluster_features_by_group(
        df=merged_df,
        group_col=smoking_col,
        title='Smoking',
        cluster_cols=cluster_feature_cols,
        min_group_n=5,
        top_n_features=6,
    )

    histology_col = sa._find_column(merged_df, ['Histology', 'histology'])
    histology_stats = sa.compare_cluster_features_by_group(
        df=merged_df,
        group_col=histology_col,
        title='Histology',
        cluster_cols=cluster_feature_cols,
        min_group_n=5,
        top_n_features=6,
    )

    return {
        'tissue_origin': tissue_origin_stats,
        'distant_location': distant_location_stats,
        'stage': stage_stats,
        'smoking': smoking_stats,
        'histology': histology_stats,
    }


clinical_group_stats = run_clinical_group_comparisons(merged_df)
clinical_group_stats

## Cox Proportional Hazards Models by Broad_treatment (PFS and OS)

This section fits lasso-regularized Cox PH models within each `Broad_treatment` subgroup for both endpoints:

- Progression-Free Survival (PFS)
  - Time: `PFS_days`
  - Event: `PFS_censoring` (`1 = progression event`, `0 = censored`)
- Overall Survival (OS)
  - Time: `OS_days`
  - Event: OS censoring column (`1 = death event`, `0 = censored`)

For each endpoint and treatment subgroup, models include:

- Lasso-regularized Cox


In [ ]:
# Step 4: Cox models by treatment subgroup

cox_run_registry, cox_overview_df = run_cox_models_by_treatment(
    merged_df,
    min_subgroup_n=20,
)
cox_overview_df

## Cox Model Diagnostics and Validation (by Endpoint and Broad_treatment)

This section validates all Cox models generated above for each endpoint/subgroup combination using:

- Harrell's concordance index (C-index)
- Approximate log-likelihood and sparsity summaries
- Risk-stratified Kaplan-Meier curves from predicted risk

Higher C-index indicates better discrimination (0.5 random, 1.0 perfect).


In [ ]:
# Step 6: Cox diagnostics and risk-stratified KM

diagnostics_df, best_models = run_cox_diagnostics(
    cox_run_registry,
    max_plots=8,
)

diagnostics_df, best_models

## Univariate Cox Follow-up for Lasso-Selected Features

This section runs one-feature-at-a-time Cox models for every feature with a non-zero coefficient in the lasso model, separately for each endpoint and `Broad_treatment` subgroup.

Reported columns include univariate hazard ratio, p-value, and 95% CI.


In [ ]:
# Step 7: univariate Cox for lasso-selected features

univariate_lasso_df, univariate_lasso_skipped = run_univariate_cox_for_lasso_selected(
    cox_run_registry,
    coef_threshold=1e-8,
 )

univariate_lasso_df, univariate_lasso_skipped

## All-Feature Univariate Cox, Lasso Comparison & Forest Plots

This section runs three analyses in sequence:

1. **Univariate Cox for all features** — fits a one-feature-at-a-time Cox PH model for every covariate in the design matrix, separately per endpoint and `Broad_treatment` subgroup. Exports hazard ratios, 95% CIs, and p-values.

2. **Univariate vs. Lasso comparison** — cross-references the univariate results against the features that survived Lasso regularization. Features are classified as:
   - **both**: Lasso-selected AND univariately significant (p < 0.05)
   - **lasso_only**: Lasso-selected but not univariately significant
   - **uni_only**: Univariately significant but not Lasso-selected
   - **neither**: Neither selected nor significant

3. **Forest plots** — one plot per endpoint/treatment subgroup showing the top features ranked by univariate p-value. Each point is the hazard ratio with a horizontal 95% CI bar. A dashed vertical line at HR = 1 separates protective (HR < 1) from risk-increasing (HR > 1) features. Points are color-coded by their Lasso/univariate agreement category.


In [ ]:
# Step 8: all-feature univariate Cox, lasso comparison, and forest plots

all_uni_df, comparison_df = run_univariate_analysis_and_forest_plots(
    cox_run_registry,
    coef_threshold=1e-8,
    p_threshold=0.05,
    max_features=30,
)

all_uni_df, comparison_df